Statistikoj laŭ la plej simpla du-litera silabaro. Ĉiuj vortoj estas dividataj je silaboj el du literoj. 
Rigardu du variantojn: 

- antaŭ-vokala konsonanto (avk): ĉiuj silaboj estas **KV**
- post-vokala konsonanto (pvk): ĉiuj silaboj estas **VK**

Helpantaj konsonanto kaj vokalo:
- se mankas vokalo por formi silabon, ni aldonu neprononceblan vokalon `y` 
- se mankas konsonanto por formi silabon, ni aldonu neprononceblan konsonanton `x`

Ekzemploj:

vorto | avk | pvk
-----|-----|-----
iu   | xI xU | Ix Ux
alta | xA Ly TA | AL yT Ax
mem  | ME My | yM EM
trans | Ty RA Ny Sy | yT yR AN ys
gemaljuneguletoj | GE MA Ly JU NE GU LE TO Jy | yG EM AL yJ UN EG UL ET OJ
babiletemulegoj | BA BI LE TE MU LE GO Jy | yB AB IL ET EM UL EG OJ

La varianto *avk* estas pli oportuna por skribi prefiksojn kaj radikojn.
La varianto *pvk* estas pli oportuna por skribi sufiksojn kaj finaĵojn.

Esperanto havas 
- 5 vokalojn: `a, e, i, o, u` 
- kaj 23 konsonantojn: `b, c, ĉ, d, f, g, ĝ, h, ĥ, j, ĵ, k, l, m, n, p, r, s, ŝ, t, ŭ, v, z`

Do en ambaŭ variantoj povas esti teorie:
- 23*5: KV aŭ VK - realaj paroj
- 5: xV aŭ Vx - vokaloj sen konsonanto
- 23: Ky aŭ yK - konsonantoj sen vokalo

Sume _143 = 23\*5 + 5 + 23 = 24*6 - 1_ (minus neebla kombino `xy`). 

Ni dividu la tekston je silaboj per ambaŭ variantoj kaj komparu kiom da diversaj duliteraj silaboj reale aperas en Esperantaj tekstoj.


In [14]:
# constants and imports

from iloj.du_litera_silab_ilo import DuLiteraSilabiIlo, Silabo

from pathlib import Path

# g_input_file: str - Path to YAML file containing word frequency data
g_input_file = "./data/source1_all_words.yaml"
# g_input_file = "./data/source2_test.yaml"

# g_input_file_path: pathlib.PosixPath - Path object for generating output filenames (GLOBAL - used in later cells)
g_input_file_path = Path(g_input_file)

In [15]:
# Parse custom YAML format (word: frequency) into dict; avoids yaml library's false-conversion bug
def read_flat_yml(input_file):
    with open(input_file, "r") as f:
        lines = f.readlines()

    all_data = {}
    for line in lines:
        vorto, kvanto = line.strip().split(": ")
        all_data[vorto] = int(kvanto)

    return all_data


In [16]:
# Load input data from YAML file and display basic statistics

# g_all_words: dict[str, int] - Mapping of Esperanto words to their occurrence frequencies
g_all_words = read_flat_yml(g_input_file)
print("Length:", len(g_all_words))
print("First 10 items:", list(g_all_words.items())[:10])

Length: 49463
First 10 items: [('la', 612479), ('de', 308999), ('kaj', 219359), ('en', 146802), ('al', 96398), ('estas', 71030), ('ne', 68672), ('mi', 57988), ('por', 57813), ('li', 56887)]


In [17]:
# Write syllable breakdown of each word to CSV file

def skribu_silabojn_csv(rezultoj: list, output_file: str, rendering_func) -> None:
    """
    Writes the list of triples (word, silaboj, frequency) as CSV into output_file.
    Columns: vorto,silaboj,frekvenco
    
    Args:
        rezultoj: list of (word, syllables, frequency) tuples
        output_file: path to output CSV file
        rendering_func: function that takes a list of Silabo and returns formatted string
    """
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("vorto,silaboj,frekvenco\n")
        for vorto, silaboj, frekvenco in rezultoj:
            formatigita = rendering_func(silaboj)
            f.write(f"{vorto},{formatigita},{frekvenco}\n")
    print(f"Skribis {len(rezultoj)} vortojn en '{output_file}'")

In [18]:
# Parse all words into syllable structures
def dividu_je_silabojn(all_words: dict, dividilo) -> list:
    """
    Given all_words dict (word -> frequency),
    for each word calls the dividilo function and builds a list of triples:
    (word, array of silaboj, frequency).
    The list preserves the input order.
    
    Args:
        all_words: dict[str, int] - mapping of words to frequencies
        dividilo: function that takes a word string and returns list of Silabo
    
    Returns:
        list of (word, syllables, frequency) tuples
    """
    rezultoj = []
    for vorto, frekvenco in all_words.items():
        silaboj = dividilo(vorto)
        rezultoj.append((vorto, silaboj, frekvenco))
    return rezultoj

In [19]:
# Execute main pipeline: parse words to AVK syllables and export to CSV

ilo = DuLiteraSilabiIlo()

# g_words_with_syllables_avk: list[tuple] - List of (word, syllables_array, frequency) tuples with complete AVK syllable breakdown
g_words_with_syllables_avk = dividu_je_silabojn(g_all_words, ilo.dividu_laŭ_avk)

skribu_silabojn_csv(g_words_with_syllables_avk, 
                    str(g_input_file_path.with_stem(f"{g_input_file_path.stem}_avk").with_suffix(".csv")),
                    ilo.skribu_avk)

# g_words_with_syllables_pvk: list[tuple] - List of (word, syllables_array, frequency) tuples with complete PVK syllable breakdown
g_words_with_syllables_pvk = dividu_je_silabojn(g_all_words, ilo.dividu_laŭ_pvk)

skribu_silabojn_csv(g_words_with_syllables_pvk, 
                    str(g_input_file_path.with_stem(f"{g_input_file_path.stem}_pvk").with_suffix(".csv")),
                    ilo.skribu_pvk)

Skribis 49463 vortojn en 'data/source1_all_words_avk.csv'
Skribis 49463 vortojn en 'data/source1_all_words_pvk.csv'


In [20]:
# Calculate distinct syllables and their frequencies
def kalkulu_distinktajn_silabojn(silaboj: list) -> dict:
    """
    Given a list of triples (vorto, vorto_silaboj, frekvenco),
    returns a dict mapping each distinct syllable (Silabo tuple) to its total frequency.
    A syllable appears as many times as the word it belongs to was used.
    Silabo is immutable (NamedTuple) so it can be used as a dictionary key.
    """
    result = {}
    for _, vorto_silaboj, frekvenco in silaboj:
        for silabo in vorto_silaboj:
            # Silabo is now a NamedTuple, so it's hashable and can be used as a key directly
            result[silabo] = result.get(silabo, 0) + frekvenco
    return result

In [21]:
# Write distinct syllables to CSV with cumulative percentage milestones

def transform_silaboj_to_strings(silaboj_dict: dict, rendering_func) -> dict:
    """
    Transform a dict with Silabo keys (immutable NamedTuples) to a dict with string keys.
    
    Args:
        silaboj_dict: dict[Silabo, int] - mapping from Silabo tuples to frequencies
        rendering_func: function that takes a Silabo and returns string representation
    
    Returns:
        dict[str, int] - mapping from string representations to frequencies
    """
    result = {}
    for silabo, freq in silaboj_dict.items():
        key = rendering_func(silabo)
        result[key] = freq
    return result


def write_syllables_to_csv(dist, output_file):
    total_sil_freq = sum(dist.values())
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("silabo,frekvenco,procento\n")
        cumulative = 0
        next_threshold = 10
        for sil, freq in sorted(dist.items(), key=lambda x: x[1], reverse=True):
            pct = freq / total_sil_freq * 100
            f.write(f"{sil},{freq},{pct:.4f}\n")
            cumulative += pct
            while next_threshold <= 90 and cumulative >= next_threshold:
                f.write(
                    f"# --- ĉi tiuj silaboj kune kovras {cumulative:.4f}% (sojlo: {next_threshold}%) ---\n"
                )
                next_threshold += 10
    print(f"Skribis {len(dist)} distinktajn silabojn en '{output_file}'")


ilo = DuLiteraSilabiIlo()

# distinktaj_silaboj_avk_raw: dict[Silabo, int] - Distinct syllables as immutable tuples → frequencies
distinktaj_silaboj_avk_raw = kalkulu_distinktajn_silabojn(g_words_with_syllables_avk)

# Transform Silabo keys to string keys for CSV output
distinktaj_silaboj_avk = transform_silaboj_to_strings(distinktaj_silaboj_avk_raw, ilo.skribu_silabon_avk)

write_syllables_to_csv(
    distinktaj_silaboj_avk,
    str(
        g_input_file_path.with_stem(
            f"{g_input_file_path.stem}_distinktaj_silaboj_avk"
        ).with_suffix(".csv")
    ),
)

# distinktaj_silaboj_pvk_raw: dict[Silabo, int] - Distinct syllables as immutable tuples → frequencies
distinktaj_silaboj_pvk_raw = kalkulu_distinktajn_silabojn(g_words_with_syllables_pvk)

# Transform Silabo keys to string keys for CSV output
distinktaj_silaboj_pvk = transform_silaboj_to_strings(distinktaj_silaboj_pvk_raw, ilo.skribu_silabon_pvk)

write_syllables_to_csv(
    distinktaj_silaboj_pvk,
    str(
        g_input_file_path.with_stem(
            f"{g_input_file_path.stem}_distinktaj_silaboj_pvk"
        ).with_suffix(".csv")
    ),
)

Skribis 143 distinktajn silabojn en 'data/source1_all_words_distinktaj_silaboj_avk.csv'
Skribis 139 distinktajn silabojn en 'data/source1_all_words_distinktaj_silaboj_pvk.csv'


In [24]:
# Visual comparison of AVK and PVK syllable distributions

VOKALOJ = ["a", "e", "i", "o", "u"]
KONSONANTOJ = ["b", "c", "ĉ", "d", "f", "g", "ĝ", "h", "ĥ", "j", "ĵ", "k", "l", "m", "n", "p", "r", "s", "ŝ", "t", "ŭ", "v", "z"]

def generiĝi_ĉiuj_silaboj():
    """Generate all 143 valid Silabo combinations:
    - 5 xV (vowel without consonant)
    - 23 Ky (consonant without vowel)
    - 115 KV (all consonant-vowel combinations)
    """
    silaboj = []
    
    # xV: vowel without consonant
    for v in VOKALOJ:
        silaboj.append(Silabo(k="x", v=v))
    
    # Ky: consonant without vowel
    for k in KONSONANTOJ:
        silaboj.append(Silabo(k=k, v="y"))
    
    # KV: all consonant-vowel pairs
    for k in KONSONANTOJ:
        for v in VOKALOJ:
            silaboj.append(Silabo(k=k, v=v))
    
    return silaboj


def kompari_silabojn_avk_vs_pvk(dist_avk: dict, dist_pvk: dict, output_file: str):
    """Compare AVK and PVK syllable distributions.
    
    Args:
        dist_avk: dict[str, int] - syllables in AVK format with frequencies
        dist_pvk: dict[str, int] - syllables in PVK format with frequencies
        output_file: path to output CSV file
    """
    # Generate all valid syllables
    ĉiuj_silaboj = generiĝi_ĉiuj_silaboj()
    
    # Calculate totals
    total_avk = sum(dist_avk.values())
    total_pvk = sum(dist_pvk.values())
    
    # Build comparison rows
    rows = []
    for silabo in ĉiuj_silaboj:
        silabo_avk_str = ilo.skribu_silabon_avk(silabo)
        silabo_pvk_str = ilo.skribu_silabon_pvk(silabo)
        
        # Check if syllable exists in AVK dict
        if silabo_avk_str in dist_avk:
            count_avk = dist_avk[silabo_avk_str]
            pct_avk = f"{count_avk / total_avk * 100:.4f}%"
        else:
            count_avk = "--"
            pct_avk = "--"
        
        # Check if syllable exists in PVK dict
        if silabo_pvk_str in dist_pvk:
            count_pvk = dist_pvk[silabo_pvk_str]
            pct_pvk = f"{count_pvk / total_pvk * 100:.4f}%"
        else:
            count_pvk = "--"
            pct_pvk = "--"
        
        rows.append((silabo_avk_str, count_avk, pct_avk, silabo_pvk_str, count_pvk, pct_pvk))
    
    # Sort by AVK syllable string
    rows.sort(key=lambda x: x[0])
    
    # Write CSV
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("silabo_avk,count_avk,percent_avk,silabo_pvk,count_pvk,percent_pvk\n")
        for silabo_avk, count_avk, pct_avk, silabo_pvk, count_pvk, pct_pvk in rows:
            f.write(f"{silabo_avk},{count_avk},{pct_avk},{silabo_pvk},{count_pvk},{pct_pvk}\n")
    
    print(f"Skribis {len(rows)} silabojn en '{output_file}'")


# Generate comparison
kompari_silabojn_avk_vs_pvk(
    distinktaj_silaboj_avk,
    distinktaj_silaboj_pvk,
    str(g_input_file_path.with_stem(f"{g_input_file_path.stem}_komparo_avk_pvk").with_suffix(".csv"))
)

Skribis 143 silabojn en 'data/source1_all_words_komparo_avk_pvk.csv'


In [26]:
# Comparison of INCOMPLETE syllables only (neplenaj silaboj: k="x" or v="y")

def generiĝi_neplenajn_silabojn():
    """Generate only incomplete Silabo combinations (where k="x" or v="y"):
    - 5 xV (vowel without consonant)
    - 23 Ky (consonant without vowel)
    Total: 28 incomplete syllables
    """
    silaboj = []
    
    # xV: vowel without consonant
    for v in VOKALOJ:
        silaboj.append(Silabo(k="x", v=v))
    
    # Ky: consonant without vowel
    for k in KONSONANTOJ:
        silaboj.append(Silabo(k=k, v="y"))
    
    return silaboj


def kompari_neplenajn_silabojn_avk_vs_pvk(dist_avk: dict, dist_pvk: dict, output_file: str):
    """Compare AVK and PVK distributions for incomplete syllables only.
    
    Args:
        dist_avk: dict[str, int] - syllables in AVK format with frequencies
        dist_pvk: dict[str, int] - syllables in PVK format with frequencies
        output_file: path to output CSV file
    """
    # Generate only incomplete syllables
    neplenaj_silaboj = generiĝi_neplenajn_silabojn()
    
    # Calculate totals
    total_avk = sum(dist_avk.values())
    total_pvk = sum(dist_pvk.values())
    
    # Build comparison rows
    rows = []
    for silabo in neplenaj_silaboj:
        silabo_avk_str = ilo.skribu_silabon_avk(silabo)
        silabo_pvk_str = ilo.skribu_silabon_pvk(silabo)
        
        # Check if syllable exists in AVK dict
        if silabo_avk_str in dist_avk:
            count_avk = dist_avk[silabo_avk_str]
            pct_avk = f"{count_avk / total_avk * 100:.4f}%"
        else:
            count_avk = "--"
            pct_avk = "--"
        
        # Check if syllable exists in PVK dict
        if silabo_pvk_str in dist_pvk:
            count_pvk = dist_pvk[silabo_pvk_str]
            pct_pvk = f"{count_pvk / total_pvk * 100:.4f}%"
        else:
            count_pvk = "--"
            pct_pvk = "--"
        
        rows.append((silabo_avk_str, count_avk, pct_avk, silabo_pvk_str, count_pvk, pct_pvk))
    
    # Sort by AVK syllable string
    rows.sort(key=lambda x: x[0])
    
    # Write CSV
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("silabo_avk,count_avk,percent_avk,silabo_pvk,count_pvk,percent_pvk\n")
        for silabo_avk, count_avk, pct_avk, silabo_pvk, count_pvk, pct_pvk in rows:
            f.write(f"{silabo_avk},{count_avk},{pct_avk},{silabo_pvk},{count_pvk},{pct_pvk}\n")
    
    print(f"Skribis {len(rows)} neplenajn silabojn en '{output_file}'")


# Generate comparison for incomplete syllables only
kompari_neplenajn_silabojn_avk_vs_pvk(
    distinktaj_silaboj_avk,
    distinktaj_silaboj_pvk,
    str(g_input_file_path.with_stem(f"{g_input_file_path.stem}_komparo_neplenaj_avk_pvk").with_suffix(".csv"))
)

Skribis 28 neplenajn silabojn en 'data/source1_all_words_komparo_neplenaj_avk_pvk.csv'
